# next-moe: обучение 0.2B + 0.1B Engram в Google Colab

Самостоятельный ноутбук: исходный код встроен, клонировать или публиковать репозиторий не нужно.
Обучаем **с нуля** самый маленький профиль: 12 × 512, 16 routed experts / top-2 + shared,
GR × 4, один Engram (2/3-граммы, 8 таблиц), GDN/full GQA = 3:1, shared KV пары global слоёв.
Реализация содержит около **193.67M neural + 100.02M table = 293.69M** параметров.

**Runtime → Change runtime type → GPU**. Начните с последовательности 128 и microbatch 1.
T4 использует FP16 autocast + GradScaler, GPU с BF16 — BF16. Все resident weights остаются FP32.
Стандартный AdamW требует около **4.38 GiB только training states**; итоговый peak выше.
GPU Colab не гарантирован; доступность и скорость надо проверить на выделенном устройстве.

По умолчанию: UK/EN Wikipedia `20231101`, 1000 документов на язык, 300 optimizer steps.
Это проверка пайплайна, а не полноценный pretraining, instruction tuning или валидация 32K–256K.
После короткого запуска связные ответы не ожидаются. Wikipedia используется с условиями
[исходного датасета](https://huggingface.co/datasets/wikimedia/wikipedia) и лицензиями статей;
для распространения корпуса/модели сохраняйте происхождение и проверяйте требования лицензий.

Выполняйте ячейки сверху вниз. Режим `tiny` предназначен только для отладки; **по умолчанию full**.


In [ ]:
%pip -q install tokenizers==0.21.4 datasets==3.6.0 matplotlib

import torch, sys, json, gc, time, shutil
from pathlib import Path
print('Python:', sys.version.split()[0], 'Torch:', torch.__version__)
assert tuple(map(int, torch.__version__.split('+')[0].split('.')[:2])) >= (2, 6), 'Нужен PyTorch >= 2.6; обновите Colab runtime.'
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(props.name, round(props.total_memory / 2**30, 2), 'GiB')
    print('BF16:', torch.cuda.is_bf16_supported())


## Параметры запуска и хранение

`USE_DRIVE=True` сохраняет данные, tokenizer и checkpoints в Google Drive. Потребуется разрешить
подключение своего Drive в Colab. Для полного профиля оставьте **не менее 10 GiB свободного места**:
AdamW checkpoint около 3.28 GiB, при atomic overwrite временно существуют старый и новый файлы,
отдельный export весов занимает около 1.09 GiB. `last.pt` перезаписывается, история не разрастается.

`USE_DRIVE=False` пишет в `/content`: при завершении runtime файлы исчезнут, скачайте их заранее.
Для нового эксперимента используйте новое `RUN_NAME`. Не меняйте tokenizer/data внутри старого run.
При восстановлении `STEPS` означает общее число updates, а не количество дополнительных.


In [ ]:
PROFILE = 'full'               # 'full' = 0.2B+0.1B; 'tiny' = только проверка кода
USE_DRIVE = True
RUN_NAME = 'next-moe-0p2b-engram0p1b-test-v1'
RESUME = False                # True после прерывания runtime
STEPS = 300
SEQ_LEN = 128
MICROBATCH = 1
GRAD_ACCUM = 4
OPTIMIZER = 'adamw'           # 'adamw8bit' — optional, требует bitsandbytes/CUDA
CHECKPOINT_BLOCKS = True
DOCUMENTS_PER_LANGUAGE = 1000
VALIDATION_DOCUMENTS = 32
SOURCE = 'wikipedia'          # либо 'local': отдельные UTF-8 .txt в uk/ и en/
LOCAL_TEXT_DIR = '/content/my_texts'
SEED = 1234

assert PROFILE in ('full', 'tiny')
if PROFILE == 'full' and not torch.cuda.is_available():
    raise RuntimeError('Для полного профиля включите GPU. CPU предназначен только для tiny-теста.')
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    STORAGE = Path('/content/drive/MyDrive/next-moe')
else:
    STORAGE = Path('/content/next-moe-artifacts')
RUN_DIR = STORAGE / RUN_NAME
DATA_DIR = RUN_DIR / 'data'
CODE_DIR = Path('/content/next-moe-code')
CODE_DIR.mkdir(parents=True, exist_ok=True)
RUN_DIR.mkdir(parents=True, exist_ok=True)
if not RESUME and (RUN_DIR / 'last.pt').exists():
    raise RuntimeError('Здесь уже есть checkpoint: задайте RESUME=True или новое RUN_NAME.')
if OPTIMIZER == 'adamw8bit':
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'bitsandbytes==0.45.5'])


## Встроенная реализация

Чистый PyTorch: no-drop MoE dispatch, chunkwise GDN через triangular solve с FP32 recurrent state,
causal Engram, GR и SDPA GQA. CUDA SDPA выбирает доступный backend автоматически: FlashAttention
не гарантирован на любой GPU/dtype. Triton GDN и fused MoE здесь не реализованы, поэтому throughput
может уступать production kernels. Checkpointing поддерживает gradient через shared K/V.

Генерация ниже пересчитывает короткий prefix; persistent KV/GDN decode API пока не реализован.
Это ограничение тестового runtime, архитектура по-прежнему допускает растущий KV.


In [ ]:
SOURCES = {'next_moe/__init__.py': '"""Experimental, trainable next-moe reference implementation."""\nfrom .model import ModelConfig, NextMoE\n', 'next_moe/data.py': '"""Bounded UK/EN document ingestion, leak-free split, tokenizer and token files."""\nfrom pathlib import Path\nimport hashlib\nimport json\nimport numpy as np\nfrom tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders\n\nLANGUAGES = (\'uk\', \'en\')\nSPECIAL = [\'<pad>\', \'<bos>\', \'<eos>\', \'<unk>\']\nWIKIPEDIA_REVISION = \'b04c8d1ceb2f5cd4588862100d08de323dccfbaa\'\n\n\ndef document_sources(source=\'wikipedia\', local_dir=None):\n    if source == \'wikipedia\':\n        from datasets import load_dataset\n        # Public parquet dataset. No trust_remote_code; no full-dataset download.\n        return {lang: iter(load_dataset(\n            \'wikimedia/wikipedia\', f\'20231101.{lang}\', revision=WIKIPEDIA_REVISION,\n            split=\'train\', streaming=True))\n                for lang in LANGUAGES}\n    if source != \'local\' or local_dir is None:\n        raise ValueError(\'Use wikipedia or local with a directory containing uk/ and en/.\')\n    result = {}\n    for lang in LANGUAGES:\n        paths = sorted((Path(local_dir) / lang).glob(\'*.txt\'))\n        if len(paths) < 2:\n            raise ValueError(f\'Need at least two separate UTF-8 documents in {lang}/.\')\n        result[lang] = ({\'text\': path.read_text(encoding=\'utf-8\'), \'id\': path.name,\n                         \'url\': None, \'title\': path.name} for path in paths)\n    return result\n\n\ndef prepare_data(output, source=\'wikipedia\', local_dir=None, documents_per_language=1000,\n                 validation_documents=32, vocab_size=65536, max_chars_per_document=20000,\n                 seed=1234):\n    if documents_per_language <= validation_documents or validation_documents < 1:\n        raise ValueError(\'Training and validation must both contain documents.\')\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    raw = document_sources(source, local_dir)\n    documents, seen, hashes, provenance, scanned = {}, set(), {}, {}, {}\n    for lang in LANGUAGES:\n        docs, ids, origins = [], [], []\n        scanned[lang] = 0\n        for record in raw[lang]:\n            scanned[lang] += 1\n            text = record[\'text\'].strip()\n            if len(text) < 200:\n                continue\n            full_digest = hashlib.sha256(text.encode(\'utf-8\')).hexdigest()\n            text = text[:max_chars_per_document]\n            digest = hashlib.sha256(text.encode(\'utf-8\')).hexdigest()\n            if digest in seen:\n                continue\n            seen.add(digest)\n            docs.append(text)\n            ids.append(digest)\n            origins.append({\'id\': record.get(\'id\'), \'url\': record.get(\'url\'),\n                            \'title\': record.get(\'title\'), \'sha256_used_text\': digest,\n                            \'sha256_full_stripped_text\': full_digest, \'used_characters\': len(text)})\n            if len(docs) >= documents_per_language:\n                break\n        if len(docs) < validation_documents + 1:\n            raise ValueError(f\'Too few usable {lang} documents ({len(docs)}).\')\n        # Shuffle locally with fixed seed; then split by complete documents.\n        rng = np.random.default_rng(seed)\n        order = rng.permutation(len(docs))\n        documents[lang] = {\'val\': [docs[i] for i in order[:validation_documents]],\n                           \'train\': [docs[i] for i in order[validation_documents:]]}\n        hashes[lang] = {\'val\': [ids[i] for i in order[:validation_documents]],\n                        \'train\': [ids[i] for i in order[validation_documents:]]}\n        provenance[lang] = {\'val\': [origins[i] for i in order[:validation_documents]],\n                            \'train\': [origins[i] for i in order[validation_documents:]]}\n    tokenizer = Tokenizer(models.BPE(unk_token=\'<unk>\'))\n    # ByteLevel preserves Ukrainian letters, punctuation and case; no strip accents.\n    tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)\n    tokenizer.decoder = decoders.ByteLevel()\n    trainer = trainers.BpeTrainer(vocab_size=vocab_size, min_frequency=2,\n                                  special_tokens=SPECIAL,\n                                  initial_alphabet=pre_tokenizers.ByteLevel.alphabet())\n    # Equal document budgets for both languages, validation excluded.\n    tokenizer.train_from_iterator((doc for lang in LANGUAGES for doc in documents[lang][\'train\']),\n                                  trainer=trainer)\n    tokenizer.save(str(output / \'tokenizer.json\'))\n    metadata = {\'source\': source, \'source_config\': \'20231101.{uk,en}\' if source == \'wikipedia\' else None,\n                \'source_dataset\': \'wikimedia/wikipedia\' if source == \'wikipedia\' else None,\n                \'source_revision\': WIKIPEDIA_REVISION if source == \'wikipedia\' else None,\n                \'requested_documents_per_language\': documents_per_language,\n                \'validation_documents_per_language\': validation_documents,\n                \'max_characters_per_document\': max_chars_per_document,\n                \'min_characters_per_document\': 200, \'scanned_documents\': scanned,\n                \'document_counts\': {lang: {split: len(documents[lang][split])\n                                          for split in (\'train\', \'val\')} for lang in LANGUAGES},\n                \'seed\': seed, \'document_hashes\': hashes, \'requested_vocab\': vocab_size,\n                \'actual_vocab\': tokenizer.get_vocab_size(), \'normalization\': \'none; byte-level BPE\',\n                \'special_ids\': {s: tokenizer.token_to_id(s) for s in SPECIAL}, \'tokens\': {}}\n    for lang in LANGUAGES:\n        for split in (\'train\', \'val\'):\n            ids = []\n            for doc in documents[lang][split]:\n                ids.extend(tokenizer.encode(doc).ids)\n                ids.append(tokenizer.token_to_id(\'<eos>\'))\n            np.asarray(ids, dtype=np.int32).tofile(output / f\'{split}_{lang}.bin\')\n            metadata[\'tokens\'][f\'{split}_{lang}\'] = len(ids)\n    (output / \'metadata.json\').write_text(json.dumps(metadata, ensure_ascii=False, indent=2))\n    (output / \'documents.json\').write_text(json.dumps(provenance, ensure_ascii=False, indent=2))\n    print(json.dumps({key: metadata[key] for key in (\'actual_vocab\', \'tokens\', \'special_ids\')}, indent=2))\n    return metadata\n\n\nclass TokenBatches:\n    """50/50 UK/EN sampling, no padding; CPU RNG state is checkpointed."""\n    def __init__(self, folder, sequence_length=128, microbatch=1, seed=1234):\n        import torch\n        self.torch = torch\n        self.length, self.batch = sequence_length, microbatch\n        self.rng = torch.Generator().manual_seed(seed)\n        self.arrays = {(split, lang): np.memmap(Path(folder) / f\'{split}_{lang}.bin\',\n                                              mode=\'r\', dtype=np.int32)\n                       for split in (\'train\', \'val\') for lang in LANGUAGES}\n        for key, array in self.arrays.items():\n            if len(array) <= sequence_length:\n                raise ValueError(f\'{key} needs more than {sequence_length} tokens.\')\n\n    def get(self, split, device, language=None, generator=None):\n        torch = self.torch\n        gen = self.rng if generator is None else generator\n        xs, ys = [], []\n        for _ in range(self.batch):\n            lang = language or LANGUAGES[int(torch.randint(2, (1,), generator=gen))]\n            array = self.arrays[(split, lang)]\n            start = int(torch.randint(len(array) - self.length, (1,), generator=gen))\n            row = torch.from_numpy(np.array(array[start:start + self.length + 1], dtype=np.int64))\n            xs.append(row[:-1]); ys.append(row[1:])\n        return torch.stack(xs).to(device), torch.stack(ys).to(device)\n', 'next_moe/model.py': '"""Trainable v0.3 reference: GR + causal Engram + GDN/GQA + no-drop MoE.\n\nThe chunkwise GDN is an exact triangular reformulation of the recurrence,\nnot a production Triton kernel. Generation recomputes the prefix (no KV API).\n"""\nfrom dataclasses import asdict, dataclass, field\nimport math\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom torch.utils.checkpoint import checkpoint\n\n\n@dataclass\nclass ModelConfig:\n    vocab_size: int = 65536\n    layers: int = 12\n    dim: int = 512\n    experts: int = 16\n    top_k: int = 2\n    expert_dim: int = 448\n    branches: int = 4\n    rank: int = 64\n    q_heads: int = 8\n    kv_heads: int = 2\n    head_dim: int = 64\n    gdn_k_heads: int = 4\n    gdn_v_heads: int = 8\n    gdn_key_dim: int = 64\n    gdn_value_dim: int = 64\n    table_rows: list[int] = field(default_factory=lambda: [390647, 390653, 390671, 390673,\n                                                          390703, 390707, 390721, 390727])\n    row_dim: int = 32\n    full_every: int = 4\n    kv_group: int = 2\n    memory_layer: int = 1  # zero-based: decoder block 2\n    gdn_chunk: int = 32\n    rope_theta: float = 10000.0\n    eos_id: int = 2\n    pad_id: int = 0\n    checkpoint_blocks: bool = True\n\n    def __post_init__(self):\n        if self.layers % self.full_every or not 0 <= self.memory_layer < self.layers:\n            raise ValueError(\'Layers must end with a full layer; memory layer must exist.\')\n        if self.q_heads % self.kv_heads or self.gdn_v_heads % self.gdn_k_heads:\n            raise ValueError(\'Head ratios must be integral.\')\n        if not 1 <= self.top_k <= self.experts or len(self.table_rows) != 8:\n            raise ValueError(\'Invalid expert or Engram configuration.\')\n        if self.kv_group != 2 or self.head_dim % 2 or self.gdn_chunk < 1:\n            raise ValueError(\'This reference supports KV pairs and even RoPE dimensions.\')\n\n    @classmethod\n    def tiny(cls):\n        """Only for correctness tests; not the 0.2B+0.1B profile."""\n        return cls(vocab_size=512, layers=8, dim=32, experts=4, top_k=2,\n                   expert_dim=24, rank=4, q_heads=4, kv_heads=2, head_dim=8,\n                   gdn_k_heads=2, gdn_v_heads=4, gdn_key_dim=8, gdn_value_dim=8,\n                   table_rows=[101, 103, 107, 109, 113, 127, 131, 137], row_dim=4,\n                   gdn_chunk=4)\n\n    def to_dict(self):\n        return asdict(self)\n\n\nclass RMSNorm(nn.Module):\n    def __init__(self, shape, eps=1e-6):\n        super().__init__()\n        self.weight = nn.Parameter(torch.ones(shape))\n        self.eps = eps\n\n    def forward(self, x):\n        y = x.float() * torch.rsqrt(x.float().square().mean(-1, keepdim=True) + self.eps)\n        return (y * self.weight.float()).to(x.dtype)\n\n\nclass GatedResidual(nn.Module):\n    def __init__(self, cfg, write=True):\n        super().__init__()\n        self.n, self.d = cfg.branches, cfg.dim\n        self.norm = RMSNorm((self.n, self.d))\n        self.down = nn.Linear(self.n * self.d, cfg.rank, bias=False)\n        self.up = nn.Linear(cfg.rank, self.n * self.d, bias=False)\n        self.write = nn.Linear(self.n * self.d, self.n, bias=False) if write else None\n\n    def read(self, r):\n        normalized = self.norm(r)\n        z = normalized.flatten(-2)\n        gate = self.up(F.silu(self.down(z) / self.n)).sigmoid().reshape_as(r)\n        x = (gate * normalized).mean(-2)\n        s = 2 * (self.write(z) / self.n).sigmoid() if self.write is not None else None\n        return x, s\n\n    @staticmethod\n    def add(r, y, s):\n        return r + s.unsqueeze(-1).to(r.dtype) * y.unsqueeze(-2).to(r.dtype)\n\n\nclass CausalConv(nn.Module):\n    def __init__(self, width, kernel=4, dilation=1):\n        super().__init__()\n        self.left = (kernel - 1) * dilation\n        self.conv = nn.Conv1d(width, width, kernel, groups=width,\n                              dilation=dilation, bias=False)\n\n    def forward(self, x):\n        return self.conv(F.pad(x.transpose(1, 2), (self.left, 0))).transpose(1, 2)\n\n\nclass Engram(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.cfg = cfg\n        self.tables = nn.ModuleList([nn.Embedding(rows, cfg.row_dim) for rows in cfg.table_rows])\n        m, n, d = 8 * cfg.row_dim, cfg.branches, cfg.dim\n        self.value = nn.Linear(m, d, bias=False)\n        self.key = nn.Linear(m, n * d, bias=False)\n        self.query_norm, self.key_norm, self.conv_norm = [RMSNorm((n, d)) for _ in range(3)]\n        self.conv = CausalConv(n * d, kernel=4, dilation=3)\n\n    def addresses(self, tokens):\n        # Token IDs are kept unchanged: no accent stripping or case folding.\n        # Reset suffix lookup at EOS; left padding is deterministic at sample starts.\n        lag1 = F.pad(tokens[:, :-1], (1, 0), value=self.cfg.pad_id)\n        lag2 = F.pad(tokens[:, :-2], (2, 0), value=self.cfg.pad_id)[:, :tokens.shape[1]]\n        reset = tokens.eq(self.cfg.eos_id) | lag1.eq(self.cfg.eos_id)\n        lag1 = torch.where(reset, self.cfg.pad_id, lag1)\n        lag2 = torch.where(reset | lag2.eq(self.cfg.eos_id), self.cfg.pad_id, lag2)\n        hashes = []\n        for i, rows in enumerate(self.cfg.table_rows):\n            # Small integer coefficients keep arithmetic far below int64 overflow.\n            code = (tokens + 1) * (1009 + 104 * i) + (lag1 + 1) * (9176 + 62 * i)\n            if i >= 4:\n                code = code + (lag2 + 1) * (65537 + 50 * i)\n            hashes.append(code.remainder(rows))\n        return hashes\n\n    def forward(self, tokens, r):\n        e = torch.cat([table(ids) for table, ids in zip(self.tables, self.addresses(tokens))], -1)\n        e = e.to(r.dtype)\n        v = self.value(e)\n        key = self.key(e).reshape_as(r)\n        score = (self.query_norm(r).float() * self.key_norm(key).float()).sum(-1) / math.sqrt(self.cfg.dim)\n        transformed = score.sign() * score.abs().clamp_min(1e-6).sqrt()\n        u = transformed.sigmoid().unsqueeze(-1).to(r.dtype) * v.unsqueeze(-2)\n        conv = self.conv(self.conv_norm(u).flatten(-2)).reshape_as(r)\n        return r + u.to(r.dtype) + F.silu(conv).to(r.dtype)\n\n\ndef delta_chunk(q, k, v, log_alpha, beta, state):\n    """Exact gated-delta recurrence within one chunk. Shapes B,H,T,D; state B,H,K,V.\n\n    Solve write residuals against a unit-lower-triangular matrix. FP32 avoids\n    accumulation loss; no attention across future positions is performed.\n    """\n    with torch.autocast(device_type=q.device.type, enabled=False):\n        q, k, v, log_alpha, beta, state = [x.float() for x in (q, k, v, log_alpha, beta, state)]\n        length = q.shape[-2]\n        cumulative = log_alpha.cumsum(-1)\n        decay = cumulative.exp()\n        ratios = (cumulative.unsqueeze(-1) - cumulative.unsqueeze(-2)).clamp_max(0).exp()\n        kk = k @ k.transpose(-1, -2)\n        lower = (ratios * kk * beta.unsqueeze(-2)).tril(-1)\n        matrix = lower + torch.eye(length, device=q.device, dtype=torch.float32)\n        rhs = v - decay.unsqueeze(-1) * (k @ state)\n        errors = torch.linalg.solve_triangular(matrix, rhs, upper=False, unitriangular=True)\n        weights = (ratios * (q @ k.transpose(-1, -2)) * beta.unsqueeze(-2)).tril()\n        y = decay.unsqueeze(-1) * (q @ state) + weights @ errors\n        last_ratios = (cumulative[..., -1:] - cumulative).exp()\n        next_state = decay[..., -1, None, None] * state + k.transpose(-1, -2) @ (\n            errors * (last_ratios * beta).unsqueeze(-1))\n        return y, next_state\n\n\nclass GDN(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.cfg = cfg\n        k = cfg.gdn_k_heads * cfg.gdn_key_dim\n        w = cfg.gdn_v_heads * cfg.gdn_value_dim\n        self.qkv = nn.Linear(cfg.dim, 2 * k + w, bias=False)\n        self.conv = CausalConv(2 * k + w)\n        self.decay = nn.Linear(cfg.dim, cfg.gdn_v_heads, bias=False)\n        self.beta = nn.Linear(cfg.dim, cfg.gdn_v_heads, bias=False)\n        self.gate = nn.Linear(cfg.dim, w, bias=False)\n        self.out = nn.Linear(w, cfg.dim, bias=False)\n        self.norm = RMSNorm(cfg.gdn_value_dim)\n        self.A_log = nn.Parameter(torch.linspace(0, math.log(16), cfg.gdn_v_heads))\n        # Initial update time constants in [0.001, 0.1].\n        dt = torch.exp(torch.linspace(math.log(.001), math.log(.1), cfg.gdn_v_heads))\n        self.dt_bias = nn.Parameter(torch.log(torch.expm1(dt)))\n\n    def forward(self, x):\n        c = self.cfg\n        b, t, _ = x.shape\n        kw, vw = c.gdn_k_heads * c.gdn_key_dim, c.gdn_v_heads * c.gdn_value_dim\n        q, k, v = F.silu(self.conv(self.qkv(x))).split([kw, kw, vw], -1)\n        repeat = c.gdn_v_heads // c.gdn_k_heads\n        q = q.reshape(b, t, c.gdn_k_heads, c.gdn_key_dim).repeat_interleave(repeat, dim=2).transpose(1, 2)\n        k = k.reshape(b, t, c.gdn_k_heads, c.gdn_key_dim).repeat_interleave(repeat, dim=2).transpose(1, 2)\n        q, k = F.normalize(q.float(), dim=-1, eps=1e-6), F.normalize(k.float(), dim=-1, eps=1e-6)\n        v = v.reshape(b, t, c.gdn_v_heads, c.gdn_value_dim).transpose(1, 2)\n        log_alpha = -F.softplus(self.decay(x).float() + self.dt_bias) * self.A_log.exp()\n        log_alpha = log_alpha.transpose(1, 2)\n        beta = self.beta(x).float().sigmoid().transpose(1, 2)\n        state = torch.zeros(b, c.gdn_v_heads, c.gdn_key_dim, c.gdn_value_dim, device=x.device)\n        outputs = []\n        for start in range(0, t, c.gdn_chunk):\n            stop = min(start + c.gdn_chunk, t)\n            y, state = delta_chunk(q[:, :, start:stop], k[:, :, start:stop], v[:, :, start:stop],\n                                  log_alpha[:, :, start:stop], beta[:, :, start:stop], state)\n            outputs.append(y)\n        y = torch.cat(outputs, dim=2).transpose(1, 2).to(x.dtype)\n        y = self.norm(y).reshape(b, t, vw) * self.gate(x).sigmoid()\n        return self.out(y)\n\n\ndef rope(x, theta):\n    t, d = x.shape[-2:]\n    positions = torch.arange(t, device=x.device).float()\n    inverse = theta ** (-torch.arange(0, d, 2, device=x.device).float() / d)\n    angle = positions[:, None] * inverse[None, :]\n    cos, sin = angle.cos().to(x.dtype), angle.sin().to(x.dtype)\n    left, right = x[..., :d // 2], x[..., d // 2:]\n    return torch.cat([left * cos - right * sin, right * cos + left * sin], -1)\n\n\nclass GQA(nn.Module):\n    def __init__(self, cfg, owner):\n        super().__init__()\n        self.cfg, self.owner = cfg, owner\n        self.q = nn.Linear(cfg.dim, cfg.q_heads * cfg.head_dim, bias=False)\n        self.gate = nn.Linear(cfg.dim, cfg.q_heads * cfg.head_dim, bias=False)\n        self.out = nn.Linear(cfg.q_heads * cfg.head_dim, cfg.dim, bias=False)\n        self.q_norm = RMSNorm(cfg.head_dim)\n        self.k_norm = RMSNorm(cfg.head_dim) if owner else None\n        self.k = nn.Linear(cfg.dim, cfg.kv_heads * cfg.head_dim, bias=False) if owner else None\n        self.v = nn.Linear(cfg.dim, cfg.kv_heads * cfg.head_dim, bias=False) if owner else None\n\n    def forward(self, x, kv=None):\n        c = self.cfg\n        b, t, _ = x.shape\n        q = rope(self.q_norm(self.q(x).reshape(b, t, c.q_heads, c.head_dim)).transpose(1, 2), c.rope_theta)\n        if self.owner:\n            k = rope(self.k_norm(self.k(x).reshape(b, t, c.kv_heads, c.head_dim)).transpose(1, 2), c.rope_theta)\n            v = self.v(x).reshape(b, t, c.kv_heads, c.head_dim).transpose(1, 2)\n            kv = (k, v)\n        if kv is None:\n            raise ValueError(\'KV reader requires source tensors.\')\n        # Explicit repeat is portable to Colab SDPA backends; logical KV stays shared.\n        k, v = [z.repeat_interleave(c.q_heads // c.kv_heads, dim=1) for z in kv]\n        y = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=0.0)\n        y = y.transpose(1, 2).reshape(b, t, c.q_heads * c.head_dim)\n        return self.out(y * self.gate(x).sigmoid()), kv\n\n\nclass Expert(nn.Module):\n    def __init__(self, dim, hidden):\n        super().__init__()\n        self.gate = nn.Linear(dim, hidden, bias=False)\n        self.up = nn.Linear(dim, hidden, bias=False)\n        self.down = nn.Linear(hidden, dim, bias=False)\n\n    def forward(self, x):\n        return self.down(F.silu(self.gate(x)) * self.up(x))\n\n\nclass MoE(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.cfg = cfg\n        self.router = nn.Linear(cfg.dim, cfg.experts, bias=False)\n        self.experts = nn.ModuleList([Expert(cfg.dim, cfg.expert_dim) for _ in range(cfg.experts)])\n        self.shared = Expert(cfg.dim, cfg.expert_dim)\n\n    def forward(self, x):\n        shape = x.shape\n        flat = x.reshape(-1, self.cfg.dim)\n        with torch.autocast(device_type=x.device.type, enabled=False):\n            probabilities = F.linear(flat.float(), self.router.weight.float()).softmax(-1)\n        weights, selected = probabilities.topk(self.cfg.top_k, dim=-1)\n        weights = weights / weights.sum(-1, keepdim=True)\n        y = torch.zeros_like(flat)\n        for idx, expert in enumerate(self.experts):\n            token, slot = torch.where(selected == idx)\n            if token.numel():\n                out = expert(flat[token]) * weights[token, slot, None].to(x.dtype)\n                y = y.index_add(0, token, out.to(y.dtype))\n        counts = torch.bincount(selected.flatten(), minlength=self.cfg.experts).float()\n        fraction = counts / selected.numel()\n        auxiliary = self.cfg.experts * (fraction.detach() * probabilities.mean(0)).sum()\n        return (y + self.shared(flat)).reshape(shape), auxiliary, counts.detach()\n\n\nclass Block(nn.Module):\n    def __init__(self, cfg, index):\n        super().__init__()\n        self.full = (index + 1) % cfg.full_every == 0\n        self.owner = self.full and ((index + 1) // cfg.full_every - 1) % cfg.kv_group == 0\n        self.memory = Engram(cfg) if index == cfg.memory_layer else None\n        self.attn_gr, self.moe_gr = GatedResidual(cfg), GatedResidual(cfg)\n        self.mixer = GQA(cfg, self.owner) if self.full else GDN(cfg)\n        self.moe = MoE(cfg)\n\n    def forward(self, r, tokens, kv):\n        if self.memory is not None:\n            r = self.memory(tokens, r)\n        x, s = self.attn_gr.read(r)\n        if self.full:\n            y, kv = self.mixer(x, kv)\n        else:\n            y = self.mixer(x)\n        r = self.attn_gr.add(r, y, s)\n        x, s = self.moe_gr.read(r)\n        y, aux, counts = self.moe(x)\n        return self.moe_gr.add(r, y, s), kv, aux, counts\n\n\nclass NextMoE(nn.Module):\n    def __init__(self, cfg):\n        super().__init__()\n        self.config = cfg\n        self.embedding = nn.Embedding(cfg.vocab_size, cfg.dim)\n        self.blocks = nn.ModuleList([Block(cfg, i) for i in range(cfg.layers)])\n        self.final_gr = GatedResidual(cfg, write=False)\n        self.apply(self._init)\n        # Zero initial write projections => scalar write 1. Small mixer outputs\n        # keep a newly initialized multi-stream residual stable.\n        for block in self.blocks:\n            nn.init.zeros_(block.attn_gr.write.weight)\n            nn.init.zeros_(block.moe_gr.write.weight)\n            nn.init.normal_(block.mixer.out.weight, std=.02 / math.sqrt(2 * cfg.layers))\n        self.last_counts = None\n\n    @staticmethod\n    def _init(module):\n        if isinstance(module, (nn.Linear, nn.Embedding)):\n            nn.init.normal_(module.weight, std=.02)\n\n    def features(self, tokens):\n        r = self.embedding(tokens)\n        if torch.is_autocast_enabled(tokens.device.type):\n            r = r.to(torch.get_autocast_dtype(tokens.device.type))\n        r = r.unsqueeze(-2).expand(-1, -1, self.config.branches, -1).clone()\n        kv, auxiliary, counts = None, [], []\n        for block in self.blocks:\n            if self.training and self.config.checkpoint_blocks and torch.is_grad_enabled():\n                r, kv, aux, count = checkpoint(block, r, tokens, kv, use_reentrant=False)\n            else:\n                r, kv, aux, count = block(r, tokens, kv)\n            auxiliary.append(aux)\n            counts.append(count)\n        self.last_counts = torch.stack(counts)\n        return self.final_gr.read(r)[0], torch.stack(auxiliary).mean()\n\n    def forward(self, tokens, targets=None, loss_chunk=64):\n        hidden, auxiliary = self.features(tokens)\n        if targets is None:\n            return F.linear(hidden, self.embedding.weight)\n        # Chunk and checkpoint the softmax/head to avoid B*T*V live logits.\n        total = hidden.new_zeros((), dtype=torch.float32)\n        for start in range(0, tokens.shape[1], loss_chunk):\n            h = hidden[:, start:start + loss_chunk]\n            y = targets[:, start:start + loss_chunk]\n            def loss_fn(hh, yy):\n                logits = F.linear(hh, self.embedding.weight)\n                return F.cross_entropy(logits.float().reshape(-1, self.config.vocab_size),\n                                       yy.reshape(-1), reduction=\'sum\')\n            loss = checkpoint(loss_fn, h, y, use_reentrant=False) if self.training else loss_fn(h, y)\n            total = total + loss\n        return {\'ce\': total / targets.numel(), \'aux\': auxiliary}\n\n    @torch.no_grad()\n    def generate(self, tokens, max_new_tokens=64, temperature=.8, top_k=40, context=512, valid_vocab=None):\n        was_training = self.training\n        self.eval()\n        try:\n            for _ in range(max_new_tokens):\n                hidden, _ = self.features(tokens[:, -context:])\n                logits = F.linear(hidden[:, -1], self.embedding.weight).float()\n                if valid_vocab is not None:\n                    logits[:, valid_vocab:] = float(\'-inf\')\n                logits[:, self.config.pad_id] = float(\'-inf\')\n                if temperature <= 0:\n                    nxt = logits.argmax(-1, keepdim=True)\n                else:\n                    logits = logits / temperature\n                    threshold = logits.topk(min(top_k, logits.shape[-1]), dim=-1).values[:, -1:]\n                    logits = logits.masked_fill(logits < threshold, float(\'-inf\'))\n                    nxt = torch.multinomial(logits.softmax(-1), 1)\n                tokens = torch.cat([tokens, nxt], -1)\n                if tokens.shape[0] == 1 and nxt.item() == self.config.eos_id:\n                    break\n            return tokens\n        finally:\n            self.train(was_training)\n', 'next_moe/train.py': '"""Single-GPU test trainer with autocast, checkpointing, evaluation and resume."""\nfrom contextlib import nullcontext\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path\nimport hashlib\nimport json\nimport math\nimport os\nimport time\nimport torch\nfrom .data import TokenBatches\nfrom .model import ModelConfig, NextMoE\n\n\n@dataclass\nclass TrainConfig:\n    steps: int = 300  # total optimizer updates, including resumed updates\n    sequence_length: int = 128\n    microbatch: int = 1\n    accumulation: int = 4\n    learning_rate: float = 2e-4\n    warmup: int = 20\n    auxiliary_weight: float = .01\n    weight_decay: float = .1\n    grad_clip: float = 1.0\n    eval_every: int = 50\n    eval_batches: int = 4\n    save_every: int = 100\n    log_every: int = 10\n    optimizer: str = \'adamw\'  # optional \'adamw8bit\'; no CPU offload\n    seed: int = 1234\n\n\nclass Trainer:\n    def __init__(self, model_config, train_config, data_dir, output_dir, device=None, resume=None):\n        self.cfg, self.model_config = train_config, model_config\n        self.device = torch.device(device or (\'cuda\' if torch.cuda.is_available() else \'cpu\'))\n        if min(train_config.steps, train_config.accumulation, train_config.microbatch,\n               train_config.sequence_length, train_config.eval_batches, train_config.log_every,\n               train_config.eval_every, train_config.save_every) < 1:\n            raise ValueError(\'Step/batch/interval values must be positive.\')\n        if train_config.optimizer not in (\'adamw\', \'adamw8bit\'):\n            raise ValueError(\'Unknown optimizer.\')\n        self.output = Path(output_dir)\n        self.output.mkdir(parents=True, exist_ok=True)\n        self.data_dir = Path(data_dir)\n        meta = json.loads((self.data_dir / \'metadata.json\').read_text())\n        if meta[\'actual_vocab\'] > model_config.vocab_size:\n            raise ValueError(\'Tokenizer vocabulary exceeds model embedding size.\')\n        if meta[\'special_ids\'][\'<eos>\'] != model_config.eos_id or meta[\'special_ids\'][\'<pad>\'] != model_config.pad_id:\n            raise ValueError(\'Tokenizer special IDs do not match model configuration.\')\n        self.data_signature = self._data_signature()\n        torch.manual_seed(train_config.seed)\n        if self.device.type == \'cuda\':\n            torch.cuda.manual_seed_all(train_config.seed)\n        self.batches = TokenBatches(data_dir, train_config.sequence_length,\n                                   train_config.microbatch, train_config.seed)\n        self.model = NextMoE(model_config).to(self.device)\n        # FP32 resident parameters/grads; autocast computes activations in BF16/FP16.\n        # Standard AdamW has the same 16 bytes/parameter storage as BF16 + FP32 master.\n        if train_config.optimizer == \'adamw8bit\':\n            if self.device.type != \'cuda\':\n                raise ValueError(\'8-bit optimizer needs a supported CUDA GPU.\')\n            import bitsandbytes as bnb\n            self.optimizer = bnb.optim.AdamW8bit(self.model.parameters(),\n                                                lr=train_config.learning_rate,\n                                                weight_decay=train_config.weight_decay)\n        else:\n            self.optimizer = torch.optim.AdamW(self.model.parameters(), lr=train_config.learning_rate,\n                                               weight_decay=train_config.weight_decay, foreach=False)\n        self.dtype = (torch.bfloat16 if self.device.type == \'cuda\' and torch.cuda.is_bf16_supported()\n                      else torch.float16)\n        self.scaler = torch.amp.GradScaler(\'cuda\', init_scale=4096,\n                                          enabled=self.device.type == \'cuda\' and self.dtype == torch.float16)\n        self.step, self.tokens_seen = 0, 0\n        self.last_counts = None\n        if resume:\n            self.load(resume)\n        (self.output / \'model_config.json\').write_text(json.dumps(model_config.to_dict(), indent=2))\n        (self.output / \'train_config.json\').write_text(json.dumps(asdict(train_config), indent=2))\n        import shutil\n        shutil.copy2(self.data_dir / \'tokenizer.json\', self.output / \'tokenizer.json\')\n        shutil.copy2(self.data_dir / \'metadata.json\', self.output / \'data_metadata.json\')\n        print(f\'Parameters: {sum(p.numel() for p in self.model.parameters()):,}; device={self.device}; \'\n              f\'autocast={self.dtype if self.device.type == "cuda" else "off"}; step={self.step}\')\n\n    def _data_signature(self):\n        digest = hashlib.sha256()\n        for name in [\'tokenizer.json\', \'metadata.json\', \'train_uk.bin\', \'train_en.bin\', \'val_uk.bin\', \'val_en.bin\']:\n            with (self.data_dir / name).open(\'rb\') as handle:\n                while chunk := handle.read(1 << 20):\n                    digest.update(chunk)\n        return digest.hexdigest()\n\n    def autocast(self):\n        return (torch.autocast(\'cuda\', dtype=self.dtype) if self.device.type == \'cuda\' else nullcontext())\n\n    def learning_rate(self):\n        c = self.cfg\n        if self.step < c.warmup:\n            return c.learning_rate * (self.step + 1) / max(c.warmup, 1)\n        progress = (self.step - c.warmup) / max(c.steps - c.warmup, 1)\n        return c.learning_rate * (.1 + .9 * .5 * (1 + math.cos(math.pi * min(progress, 1))))\n\n    @torch.no_grad()\n    def evaluate(self):\n        self.model.eval()\n        result = {}\n        rng = torch.Generator().manual_seed(self.cfg.seed + 999)  # same windows each evaluation\n        for language in (\'uk\', \'en\'):\n            losses = []\n            for _ in range(self.cfg.eval_batches):\n                x, y = self.batches.get(\'val\', self.device, language, rng)\n                with self.autocast():\n                    losses.append(float(self.model(x, y)[\'ce\']))\n            result[f\'val_ce_{language}\'] = sum(losses) / len(losses)\n            result[f\'val_ppl_{language}\'] = math.exp(min(result[f\'val_ce_{language}\'], 20))\n        self.model.train()\n        return result\n\n    def save(self, filename=\'last.pt\'):\n        # Atomic replace: an interruption during writing preserves the previous file.\n        destination = self.output / filename\n        temporary = destination.with_suffix(\'.pt.tmp\')\n        checkpoint = {\'model\': self.model.state_dict(), \'optimizer\': self.optimizer.state_dict(),\n                      \'scaler\': self.scaler.state_dict(), \'step\': self.step, \'tokens_seen\': self.tokens_seen,\n                      \'model_config\': self.model_config.to_dict(), \'train_config\': asdict(self.cfg),\n                      \'data_signature\': self.data_signature, \'batch_rng\': self.batches.rng.get_state(),\n                      \'torch_rng\': torch.get_rng_state(),\n                      \'cuda_rng\': torch.cuda.get_rng_state_all() if self.device.type == \'cuda\' else []}\n        torch.save(checkpoint, temporary)\n        os.replace(temporary, destination)\n        print(f\'Saved {destination} ({destination.stat().st_size / 2**30:.2f} GiB)\')\n        return destination\n\n    def load(self, path):\n        saved = torch.load(path, map_location=\'cpu\', weights_only=True, mmap=True)\n        if saved[\'model_config\'] != self.model_config.to_dict():\n            raise ValueError(\'Resume requires identical model configuration.\')\n        if saved[\'data_signature\'] != self.data_signature:\n            raise ValueError(\'Resume requires exactly the same tokenizer and token corpus.\')\n        # Strict resume: only output intervals and total steps may change.\n        old = saved[\'train_config\']\n        for name in (\'optimizer\', \'sequence_length\', \'microbatch\', \'accumulation\', \'learning_rate\',\n                     \'warmup\', \'weight_decay\', \'grad_clip\', \'auxiliary_weight\', \'seed\'):\n            if old[name] != getattr(self.cfg, name):\n                raise ValueError(f\'Resume changed {name}. Start a new run instead.\')\n        self.model.load_state_dict(saved[\'model\'])\n        self.optimizer.load_state_dict(saved[\'optimizer\'])\n        self.scaler.load_state_dict(saved[\'scaler\'])\n        self.step, self.tokens_seen = saved[\'step\'], saved[\'tokens_seen\']\n        self.batches.rng.set_state(saved[\'batch_rng\'])\n        torch.set_rng_state(saved[\'torch_rng\'])\n        if self.device.type == \'cuda\' and saved[\'cuda_rng\']:\n            torch.cuda.set_rng_state_all(saved[\'cuda_rng\'])\n        print(f\'Resumed optimizer step {self.step}; changing total steps also changes the future cosine schedule.\')\n\n    def run(self):\n        c = self.cfg\n        self.model.train()\n        self.optimizer.zero_grad(set_to_none=True)\n        if self.device.type == \'cuda\':\n            torch.cuda.reset_peak_memory_stats()\n        last_log, window_tokens = time.perf_counter(), 0\n        overflow_retries = 0\n        try:\n            while self.step < c.steps:\n                lr = self.learning_rate()\n                for group in self.optimizer.param_groups:\n                    group[\'lr\'] = lr\n                ce_total, aux_total, counts = 0.0, 0.0, None\n                batch_rng_start = self.batches.rng.get_state()\n                for _ in range(c.accumulation):\n                    x, y = self.batches.get(\'train\', self.device)\n                    with self.autocast():\n                        out = self.model(x, y)\n                        loss = (out[\'ce\'] + c.auxiliary_weight * out[\'aux\']) / c.accumulation\n                    if not bool(torch.isfinite(loss)):\n                        raise FloatingPointError(\'Non-finite loss. Check data, LR and precision.\')\n                    self.scaler.scale(loss).backward()\n                    ce_total += float(out[\'ce\'].detach()) / c.accumulation\n                    aux_total += float(out[\'aux\'].detach()) / c.accumulation\n                    now_counts = self.model.last_counts\n                    counts = now_counts.clone() if counts is None else counts + now_counts\n                self.scaler.unscale_(self.optimizer)\n                norm = torch.nn.utils.clip_grad_norm_(self.model.parameters(), c.grad_clip)\n                if not bool(torch.isfinite(norm)):\n                    if self.scaler.is_enabled() and overflow_retries < 8:\n                        # unscale_ registered infs: scaler.step skips the optimizer update.\n                        # Restore sampler so the smaller scale retries the same microbatches.\n                        old_scale = self.scaler.get_scale()\n                        self.scaler.step(self.optimizer)\n                        self.scaler.update()\n                        self.optimizer.zero_grad(set_to_none=True)\n                        self.batches.rng.set_state(batch_rng_start)\n                        overflow_retries += 1\n                        print(f\'FP16 overflow: retry {overflow_retries}, scale {old_scale} -> {self.scaler.get_scale()}\')\n                        continue\n                    raise FloatingPointError(\'Persistent non-finite gradients. Reduce LR or use BF16 if available.\')\n                overflow_retries = 0\n                self.scaler.step(self.optimizer)\n                self.scaler.update()\n                self.optimizer.zero_grad(set_to_none=True)\n                self.step += 1\n                seen = c.microbatch * c.sequence_length * c.accumulation\n                self.tokens_seen += seen\n                window_tokens += seen\n                self.last_counts = counts\n                record = {\'step\': self.step, \'train_ce\': ce_total, \'router_aux\': aux_total,\n                          \'lr\': lr, \'tokens_seen\': self.tokens_seen, \'grad_norm\': float(norm),\n                          \'unselected_expert_fraction_this_step\': float((counts == 0).float().mean()),\n                          \'max_expert_to_mean_load\': float((counts.max(-1).values / counts.mean(-1).clamp_min(1)).max())}\n                if self.step % c.log_every == 0 or self.step == 1:\n                    if self.device.type == \'cuda\':\n                        torch.cuda.synchronize()\n                        record[\'peak_allocated_GiB\'] = torch.cuda.max_memory_allocated() / 2**30\n                        record[\'peak_reserved_GiB\'] = torch.cuda.max_memory_reserved() / 2**30\n                    record[\'tokens_per_second_including_interval_overhead\'] = window_tokens / max(time.perf_counter() - last_log, 1e-6)\n                    print(json.dumps(record))\n                    last_log, window_tokens = time.perf_counter(), 0\n                if self.step % c.eval_every == 0:\n                    record.update(self.evaluate())\n                    print(json.dumps({\'step\': self.step, **{k: v for k, v in record.items() if k.startswith(\'val_\')}}))\n                with (self.output / \'metrics.jsonl\').open(\'a\') as handle:\n                    handle.write(json.dumps(record) + \'\\n\')\n                if self.step % c.save_every == 0:\n                    self.save()\n        except KeyboardInterrupt:\n            # If interrupt occurred inside accumulation, partial gradients are not saved.\n            self.optimizer.zero_grad(set_to_none=True)\n            print(\'Interrupted: saving the last completed optimizer step; partial microbatch RNG may have advanced.\')\n        except torch.cuda.OutOfMemoryError:\n            print(\'CUDA OOM: reduce sequence_length/microbatch and start a NEW run; leave checkpointing enabled.\')\n            raise\n        self.save()\n        return self.evaluate()\n\n    def export_weights(self):\n        path = self.output / \'model.pt\'\n        torch.save({\'model_config\': self.model_config.to_dict(), \'model\': self.model.state_dict()}, path)\n        return path\n'}
SOURCE_HASHES = {'next_moe/__init__.py': '783e1386ddb7244ccd199a59860580bd34bc58d92c79587d7d1c39e48f1dc975', 'next_moe/data.py': '330a4f167e6e9b0f61320cb23a5b0c8a86afa9dd800e065bf39607f833aafa05', 'next_moe/model.py': '5cb9e90422b8be96aa122aa19d78b68a8769172792b6a93de6e228764dec96a9', 'next_moe/train.py': 'ee0d772b1343e387c1fc1fb7288754e524acc97f6f9ee3666b5acaffde3ed770'}
for relative, content in SOURCES.items():
    target = CODE_DIR / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content)
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
from next_moe.model import ModelConfig, NextMoE, delta_chunk
from next_moe.data import prepare_data
from next_moe.train import TrainConfig, Trainer
if RESUME and (RUN_DIR / 'source_hashes.json').exists():
    previous_hashes = json.loads((RUN_DIR / 'source_hashes.json').read_text())
    if previous_hashes != SOURCE_HASHES:
        raise RuntimeError('Исходники изменились: для strict resume нужен тот же notebook.')
(RUN_DIR / 'source_hashes.json').write_text(json.dumps(SOURCE_HASHES, indent=2))
shutil.copytree(CODE_DIR / 'next_moe', RUN_DIR / 'source', dirs_exist_ok=True)
print('Исходники сохранены:', RUN_DIR / 'source')


## Короткая проверка forward/backward

Используется отдельная tiny-модель, чтобы найти ошибки до выделения памяти полного профиля.
Проверяются causal prefix, finite loss/gradients и backward с checkpointing.
Эта модель удаляется перед настоящим обучением.


In [ ]:
torch.manual_seed(SEED)
check_device = 'cuda' if torch.cuda.is_available() else 'cpu'
small = NextMoE(ModelConfig.tiny()).to(check_device)
probe = torch.randint(4, 512, (1, 9), device=check_device)
small.eval()
with torch.no_grad():
    full_logits = small(probe)
    prefix_logits = small(probe[:, :5])
torch.testing.assert_close(full_logits[:, :5], prefix_logits, atol=2e-4, rtol=2e-3)
small.train()
probe_loss = small(probe[:, :-1], probe[:, 1:])
(probe_loss['ce'] + .01 * probe_loss['aux']).backward()
assert torch.isfinite(probe_loss['ce'])
assert all(torch.isfinite(p.grad).all() for p in small.parameters() if p.grad is not None)
print('Forward/backward и causal prefix: OK; test loss:', float(probe_loss['ce'].detach()))
del small, probe, probe_loss, full_logits, prefix_logits
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()


## UK/EN Wikipedia → train/validation → собственный tokenizer

Streaming читает ограниченное число документов из публичного snapshot, не загружает всю Википедию.
Полные документы дедуплицируются и разделяются до обучения tokenizer; validation не участвует
в BPE training. Сэмплирование при тренировке 50/50 по языкам. Byte-level BPE сохраняет і/ї/й/є/ґ,
регистр и пунктуацию. Число реально обученных BPE tokens может быть меньше 65 536; размер embedding
полного профиля всё равно 65 536, чтобы сохранять выбранный размер модели.

Статьи ограничены 20 000 символами; это намеренно маленький тестовый корпус. Статьи объединяются
через EOS, attention/GDN не сбрасываются на границе документа; Engram suffix hashing сбрасывается.
Начало sampled sequence не получает состояние предшествующего фрагмента.
Если источник недоступен, выберите `SOURCE='local'` и положите документы в `LOCAL_TEXT_DIR/uk` и `/en`.
Сетевые ошибки не заменяются скрытым toy corpus.


In [ ]:
model_cfg = ModelConfig() if PROFILE == 'full' else ModelConfig.tiny()
model_cfg.checkpoint_blocks = CHECKPOINT_BLOCKS
if not (DATA_DIR / 'metadata.json').exists():
    if RESUME:
        raise RuntimeError('Для resume нужны исходные tokenizer и token files; не пересоздавайте их.')
    prepare_data(DATA_DIR, source=SOURCE, local_dir=LOCAL_TEXT_DIR,
                 documents_per_language=DOCUMENTS_PER_LANGUAGE,
                 validation_documents=VALIDATION_DOCUMENTS,
                 vocab_size=model_cfg.vocab_size, seed=SEED)
else:
    print('Используем уже сохранённые данные; изменение настроек их не пересоздаёт.')
metadata = json.loads((DATA_DIR / 'metadata.json').read_text())
print(json.dumps({k: metadata[k] for k in ('source', 'source_revision', 'document_counts', 'actual_vocab', 'tokens')}, indent=2))
from tokenizers import Tokenizer
tokenizer = Tokenizer.from_file(str(DATA_DIR / 'tokenizer.json'))
for text in ['Україна: і, ї, й, є, ґ; пам’ять.', 'Knowledge and language.']:
    assert tokenizer.decode(tokenizer.encode(text).ids) == text
print('UK/EN tokenizer roundtrip: OK')


## Создание модели и проверка памяти

Веса и градиенты FP32; autocast выполняет вычисления в FP16/BF16. Standard AdamW имеет
16 bytes/parameter постоянных состояний после первого update. AdamW8bit — ориентировочно
10 bytes/parameter при этих FP32 weights/grads, плюс quantization metadata и small FP32 tensors.
На T4 GradScaler уменьшает scale при overflow и повторяет те же микробатчи без учёта пропущенного
optimizer update. Устойчивые non-finite gradients останавливают запуск.

Не начинайте с 32K: это ещё не production GDN/MoE implementation. При OOM попробуйте `SEQ_LEN=64`
или 32, оставьте microbatch 1 и checkpointing; для изменения формы тренировочного run выберите
новое имя. Gradient accumulation увеличивает effective batch без хранения всех микробатчей одновременно.


In [ ]:
train_cfg = TrainConfig(steps=STEPS, sequence_length=SEQ_LEN, microbatch=MICROBATCH,
                        accumulation=GRAD_ACCUM, optimizer=OPTIMIZER, seed=SEED,
                        eval_every=50, save_every=100, log_every=10)
resume_path = RUN_DIR / 'last.pt' if RESUME else None
if RESUME and not resume_path.exists(): raise FileNotFoundError(resume_path)
trainer = Trainer(model_cfg, train_cfg, DATA_DIR, RUN_DIR, resume=resume_path)
parameter_count = sum(p.numel() for p in trainer.model.parameters())
print('Training state lower bound, GiB:', parameter_count * (16 if OPTIMIZER == 'adamw' else 10) / 2**30)
print('Tokens / optimizer step:', SEQ_LEN * MICROBATCH * GRAD_ACCUM)
print('Planned new tokens:', max(STEPS - trainer.step, 0) * SEQ_LEN * MICROBATCH * GRAD_ACCUM)
print('Initial validation:', trainer.evaluate())


## Обучение

Метрики: training CE, UK/EN validation CE/perplexity, router load, gradient norm, tokens/s и peak VRAM.
Throughput включает overhead между логами, включая evaluation/checkpoint saves; это не isolated kernel benchmark.
В начале CUDA memory растёт из-за lazy allocation Adam states. Не путайте initial allocation с steady-state peak.

`last.pt` сохраняется каждые 100 updates и в конце; Ctrl+C пытается сохранить последний завершённый update.
При жёстком отключении runtime сохранится только последний успешно записанный checkpoint.
После такого отключения запустите ноутбук заново с тем же RUN_NAME и `RESUME=True`.
При увеличении STEPS меняется будущая cosine LR schedule; прежняя история обучения сохраняется.


In [ ]:
start = time.perf_counter()
final_metrics = trainer.run()
print('Elapsed minutes:', round((time.perf_counter() - start) / 60, 2))
print('Final validation:', final_metrics)
weights_path = trainer.export_weights()
print('Weights:', weights_path)
# Record exact runtime package versions for this run.
import subprocess
(RUN_DIR / 'pip-freeze.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))


## Графики и загрузка экспертов

Validation curve важнее training loss для обнаружения переобучения. Баланс router не доказывает
качество экспертов. Метрика unselected относится к одному update, а не ко всему обучению.


In [ ]:
import matplotlib.pyplot as plt
records = [json.loads(line) for line in (RUN_DIR / 'metrics.jsonl').read_text().splitlines()]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot([r['step'] for r in records], [r['train_ce'] for r in records], label='train')
for lang in ['uk', 'en']:
    validation = [r for r in records if f'val_ce_{lang}' in r]
    ax[0].plot([r['step'] for r in validation], [r[f'val_ce_{lang}'] for r in validation], label=f'val {lang}')
ax[0].set(xlabel='Optimizer step', ylabel='Cross entropy'); ax[0].legend()
counts = trainer.last_counts
if counts is not None:
    image = ax[1].imshow(counts.float().cpu().numpy(), aspect='auto')
    ax[1].set(xlabel='Routed expert', ylabel='Layer', title='Assignments: last update')
    fig.colorbar(image, ax=ax[1])
plt.tight_layout(); plt.show()


## Тестовая генерация

Это обычное продолжение текста, не чат-модель. Тестовый runtime пересчитывает prefix на каждом токене
и оставляет последние SEQ_LEN tokens; это не реализация long-context inference cache.
После 300 updates случайный или бессвязный результат нормален: главное — проверить корректность
обучения, finite gradients и изменение loss. Качество языка и знаний требует гораздо большего корпуса
и вычислений, а instruct поведение — отдельного этапа обучения.


In [ ]:
for prompt in ['Україна — це', 'The history of science']:
    ids = [tokenizer.token_to_id('<bos>')] + tokenizer.encode(prompt).ids
    tokens = torch.tensor([ids], device=trainer.device)
    with trainer.autocast():
        generated = trainer.model.generate(tokens, max_new_tokens=48, temperature=.8, top_k=40,
                                           context=SEQ_LEN, valid_vocab=tokenizer.get_vocab_size())
    print(tokenizer.decode(generated[0].tolist()))
    print()


## Экспорт и восстановление для inference

В Drive уже лежат `model.pt`, `tokenizer.json`, `model_config.json`, `last.pt`, метрики и исходники.
`model.pt` нужен для inference; `last.pt` дополнительно содержит optimizer/scaler/RNG для продолжения.
На восстановление нужен тот же tokenizer; не retrain BPE для существующих Engram rows.
Загружайте только свои доверенные checkpoints. Пример отдельной загрузки ниже выключен по умолчанию,
чтобы не создавать вторую копию большой модели на GPU.


In [ ]:
RELOAD_FOR_INFERENCE = False
if RELOAD_FOR_INFERENCE:
    del trainer
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    payload = torch.load(RUN_DIR / 'model.pt', map_location='cpu', weights_only=True, mmap=True)
    restored = NextMoE(ModelConfig(**payload['model_config']))
    restored.load_state_dict(payload['model'])
    restored = restored.to('cuda' if torch.cuda.is_available() else 'cpu').eval()
    del payload
    print('Restored model:', sum(p.numel() for p in restored.parameters()))

# Для USE_DRIVE=False можно скачать веса вручную; большой файл может занять время.
DOWNLOAD_WEIGHTS = False
if DOWNLOAD_WEIGHTS:
    from google.colab import files
    files.download(str(RUN_DIR / 'model.pt'))
    files.download(str(RUN_DIR / 'tokenizer.json'))
